# 💧 LFM2 - DPO with TRL

This tutorial demonstrates how to fine-tune our LFM2 models, e.g. [`LiquidAI/LFM2.5-1.2B-Instruct`](https://huggingface.co/LiquidAI/LFM2.5-1.2B-Instruct), using the TRL library.

Follow along if it's your first time using trl, or take single code snippets for your own workflow

## 🎯 What You'll Find:
- **DPO** (Direct Preference Optimization) - Align with human preferences  

## 📋 Prerequisites:
- **GPU Runtime**: Select GPU in `Runtime` → `Change runtime type`
- **Hugging Face Account**: For accessing models and datasets



# 📦 Installation & Setup

First, let's install all the required packages:


In [ ]:
!uv pip install transformers==4.54.0 trl>=0.18.2 peft>=0.15.2

Let's now verify the packages are installed correctly

In [ ]:
import torch
import transformers
import trl
import os
os.environ["WANDB_DISABLED"] = "true"

print(f"📦 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers version: {transformers.__version__}")
print(f"📊 TRL version: {trl.__version__}")

📦 PyTorch version: 2.6.0+cu124
🤗 Transformers version: 4.54.0
📊 TRL version: 0.19.1


# Loading the model from Transformers 🤗



In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "LiquidAI/LFM2.5-1.2B-Instruct"

print("📚 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("🧠 Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto",
)

print("✅ Local model loaded successfully!")
print(f"🔢 Parameters: {model.num_parameters():,}")
print(f"📖 Vocab size: {len(tokenizer)}")
print(f"💾 Model size: ~{model.num_parameters() * 2 / 1e9:.1f} GB (bfloat16)")

📚 Loading tokenizer...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


🧠 Loading model...
✅ Local model loaded successfully!
🔢 Parameters: 1,170,340,608
📖 Vocab size: 64400
💾 Model size: ~2.3 GB (bfloat16)


# 🎯 Direct Preference Optimization (DPO + LoRA)

DPO aligns the model with human preferences by learning from preference pairs (chosen vs rejected responses). This typically follows SFT training.

DPO might be too compute heavy if you're running on one of the free-tier colab GPUs. Hence we use LoRA (Low-Rank Adaptation) to finetune the model by only training a small number of additional parameters. Perfect for limited compute resources!

## Load a DPO Dataset

We will use [mlabonne/orpo-dpo-mix-40k](https://huggingface.co/datasets/mlabonne/orpo-dpo-mix-40k), limiting ourselves to the first 2k samples for brevity. Feel free to change the limit by changing the slicing index in the parameter `split`. The size of the validation data can be adjusted by changing `test_size`.

In [ ]:
from datasets import load_dataset

print("📥 Loading DPO dataset...")

dataset_dpo = load_dataset("mlabonne/orpo-dpo-mix-40k", split="train[:2000]")
dataset_dpo = dataset_dpo.train_test_split(test_size=0.1, seed=42)
train_dataset_dpo, eval_dataset_dpo = dataset_dpo['train'], dataset_dpo['test']

print("✅ DPO Dataset loaded:")
print(f"   📚 Train samples: {len(train_dataset_dpo)}")
print(f"   🧪 Eval samples: {len(eval_dataset_dpo)}")

sample = train_dataset_dpo[0]
print("\n📝 Single Sample:")
print(f"   Prompt: {sample['prompt'][:100]}...")
print(f"   ✅ Chosen: {sample['chosen'][:100]}...")
print(f"   ❌ Rejected: {sample['rejected'][:100]}...")

📥 Loading DPO dataset...
✅ DPO Dataset loaded:
   📚 Train samples: 1800
   🧪 Eval samples: 200

📝 Single Sample:
   Prompt: Classify the following instruments into their respective families (brass, strings, woodwinds, or per...
   ✅ Chosen: [{'content': 'Classify the following instruments into their respective families (brass, strings, woodwinds, or percussion): Didgeridoo, Cuica, Euphonium, Guzheng', 'role': 'user'}, {'content': 'The classification of the mentioned instruments is as follows:\n\n1. Didgeridoo: Brass\n2. Cuica: Percussion\n3. Euphonium: Brass\n4. Guzheng: Strings', 'role': 'assistant'}, {'content': 'Explain the construction and sound production process of each instrument mentioned, and how these processes relate to their classification in their respective families.', 'role': 'user'}, {'content': "1. Didgeridoo: The didgeridoo is traditionally made from eucalyptus trees which have been naturally hollowed out by termites. The mouthpiece can be made of beeswax or shaped di

## Wrap the model with PEFT

We specify target modules that will be finetuned while the rest of the models weights remains frozen. Feel free to modify the `r` (rank) value:
- higher -> better approximation of full-finetuning
- lower -> needs even less compute resources

You can skip this part if you have a premium GPU and want to go for a full finetune.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

GLU_MODULES = ["w1", "w2", "w3"]
MHA_MODULES = ["q_proj", "k_proj", "v_proj", "out_proj"]
CONV_MODULES = ["in_proj", "out_proj"]

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8,  # <- lower values = fewer parameters
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=GLU_MODULES + MHA_MODULES + CONV_MODULES,
    bias="none",
    modules_to_save=None,
)

lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()

print("✅ LoRA configuration applied!")
print(f"🎛️  LoRA rank: {lora_config.r}")
print(f"📊 LoRA alpha: {lora_config.lora_alpha}")
print(f"🎯 Target modules: {lora_config.target_modules}")

trainable params: 5,554,176 || all params: 1,175,894,784 || trainable%: 0.4723
✅ LoRA configuration applied!
🎛️  LoRA rank: 8
📊 LoRA alpha: 16
🎯 Target modules: {'w2', 'out_proj', 'in_proj', 'q_proj', 'k_proj', 'w3', 'v_proj', 'w1'}


## Launch Training

We are now ready to launch a DPO run with `DPOTrainer`, feel free to modify `DPOConfig` to play around with different configurations.

Replace `lora_model` with `model` if you have a premium-tier in colab and want to run a full finetune.

In [ ]:
from trl import DPOConfig, DPOTrainer

# DPO Training configuration
dpo_config = DPOConfig(
    output_dir="./lfm2-dpo",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    learning_rate=1e-6,
    lr_scheduler_type="linear",
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=False # <- not all colab GPUs support bf16
)

# Create DPO trainer
print("🏗️  Creating DPO trainer...")
dpo_trainer = DPOTrainer(
    model=lora_model,
    args=dpo_config,
    train_dataset=train_dataset_dpo,
    eval_dataset=eval_dataset_dpo,
    processing_class=tokenizer,
)

# Start DPO training
print("\n🚀 Starting DPO training...")
dpo_trainer.train()

print("🎉 DPO training completed!")

# Save the DPO model
dpo_trainer.save_model()
print(f"💾 DPO model saved to: {dpo_config.output_dir}")

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


🏗️  Creating DPO trainer...


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.



🚀 Starting DPO training...


Epoch,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
0,0.703900,0.697031,0.009028,0.002252,0.235000,0.006835,-566.400024,-738.239990,-2.136563,-2.148125


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


🎉 DPO training completed!
💾 DPO model saved to: ./lfm2-dpo


## Save merged model

If you have used LoRA. Merge the extra weights learned with LoRA back into the model to obtain a "normal" model checkpoint.

In [ ]:
print("\n🔄 Merging LoRA weights...")
merged_model = lora_model.merge_and_unload()
merged_model.save_pretrained("./lfm2-lora-merged")
tokenizer.save_pretrained("./lfm2-lora-merged")
print("💾 Merged model saved to: ./lfm2-lora-merged")


🔄 Merging LoRA weights...
💾 Merged model saved to: ./lfm2-lora-merged
